<a href="https://colab.research.google.com/github/srishtiii28/flight_delay_predictor/blob/main/flight_delay_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving Air-Clean.csv to Air-Clean.csv


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
df = pd.read_csv("Air-Clean.csv")
df.head()

,airline,flightNumber,origin,destination,daysOfWeek,scheduledDepartureTime,scheduledArrivalTime,timezone,validFrom,validTo,lastUpdated
0,GoAir,572,Goa,Hyderabad,"Thursday,Friday,Saturday",17:25,18:40,2021-03-27,2020-10-25,2021-03-27,2025-07-16
1,GoAir,301,Goa,Delhi,Saturday,12:00,14:45,2022-10-23,2022-03-27,2022-10-23,2025-07-16
2,GoAir,733,Goa,Nagpur,Saturday,12:10,13:40,2020-10-18,2020-03-29,2020-10-18,2025-07-16
3,GoAir,667,Goa,Lucknow,"Sunday,Monday,Tuesday,Wednesday,Thursday,Frida...",06:10,09:10,2019-10-26,2019-05-01,2019-10-26,2025-07-16
4,GoAir,247,Goa,Mumbai,Saturday,11:00,12:30,2022-03-20,2021-10-31,2022-03-20,2025-07-16


In [4]:
print(df.columns)
df.info()
df.isnull().sum()

Index(['airline', 'flightNumber', 'origin', 'destination', 'daysOfWeek',
       'scheduledDepartureTime', 'scheduledArrivalTime', 'timezone',
       'validFrom', 'validTo', 'lastUpdated'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33734 entries, 0 to 33733
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   airline                 33734 non-null  object
 1   flightNumber            33734 non-null  object
 2   origin                  33734 non-null  object
 3   destination             33734 non-null  object
 4   daysOfWeek              33734 non-null  object
 5   scheduledDepartureTime  33734 non-null  object
 6   scheduledArrivalTime    33734 non-null  object
 7   timezone                33734 non-null  object
 8   validFrom               33734 non-null  object
 9   validTo                 33734 non-null  object
 10  lastUpdated             33734 non-null  object
dtyp

,0
airline,0
flightNumber,0
origin,0
destination,0
daysOfWeek,0
scheduledDepartureTime,0
scheduledArrivalTime,0
timezone,0
validFrom,0
validTo,0


In [6]:
df = df.drop_duplicates()
df = df.dropna()

In [7]:
def convert_to_minutes(time):
    try:
        time = str(time).zfill(4)
        return int(time[:2]) * 60 + int(time[2:])
    except:
        return np.nan

In [9]:
df['Dep_min'] = df['scheduledDepartureTime'].apply(convert_to_minutes)
df['Arr_min'] = df['scheduledArrivalTime'].apply(convert_to_minutes)

In [10]:
df['Delay'] = df['Arr_min'] - df['Dep_min']

In [12]:
df['Delay'] = df['Delay'].apply(lambda x: x if x >= 0 else x + 1440)

In [13]:
df['DELAYED'] = df['Delay'].apply(lambda x: 1 if x > 15 else 0)

In [15]:
df['Dep_hour'] = df['Dep_min'] // 60

In [16]:
df['Peak'] = df['Dep_hour'].apply(lambda x: 1 if 6 <= x <= 10 or 17 <= x <= 21 else 0)

In [17]:
# Get all unique days of the week present in the 'daysOfWeek' column
all_days = set()
for index, row in df.iterrows():
    days = [day.strip() for day in str(row['daysOfWeek']).split(',')]
    for day in days:
        if day:
            all_days.add(day)

# Create new binary columns for each day of the week
for day in all_days:
    df[f'is_{day}'] = df['daysOfWeek'].apply(lambda x: 1 if day in x else 0)

# Display the head of the DataFrame with new columns
display(df[['daysOfWeek'] + [f'is_{day}' for day in sorted(list(all_days))]].head())

,daysOfWeek,is_Friday,is_Monday,is_Saturday,is_Sunday,is_Thursday,is_Tuesday,is_Wednesday
0,"Thursday,Friday,Saturday",1,0,1,0,1,0,0
1,Saturday,0,0,1,0,0,0,0
2,Saturday,0,0,1,0,0,0,0
3,"Sunday,Monday,Tuesday,Wednesday,Thursday,Frida...",1,1,1,1,1,1,1
4,Saturday,0,0,1,0,0,0,0


Next, let's process the date columns (`validFrom`, `validTo`, `lastUpdated`). Converting them to datetime objects will allow us to extract meaningful temporal features, such as the duration of the flight schedule's validity.

In [18]:
# Convert date columns to datetime objects
df['validFrom'] = pd.to_datetime(df['validFrom'])
df['validTo'] = pd.to_datetime(df['validTo'])
df['lastUpdated'] = pd.to_datetime(df['lastUpdated'])

# Calculate the duration of the schedule in days
df['schedule_duration_days'] = (df['validTo'] - df['validFrom']).dt.days

# Display the head of the DataFrame with new columns
display(df[['validFrom', 'validTo', 'lastUpdated', 'schedule_duration_days']].head())

,validFrom,validTo,lastUpdated,schedule_duration_days
0,2020-10-25,2021-03-27,2025-07-16,153
1,2022-03-27,2022-10-23,2025-07-16,210
2,2020-03-29,2020-10-18,2025-07-16,203
3,2019-05-01,2019-10-26,2025-07-16,178
4,2021-10-31,2022-03-20,2025-07-16,140


In [22]:
from sklearn.preprocessing import LabelEncoder

# List of categorical columns to process
categorical_cols_to_process = ['airline', 'origin', 'destination', 'timezone']

# Apply LabelEncoder to each categorical column, only if it exists
for col in categorical_cols_to_process:
    if col in df.columns:
        le = LabelEncoder()
        df[col + '_encoded'] = le.fit_transform(df[col])
    else:
        print(f"Warning: Column '{col}' not found in DataFrame. Skipping encoding.")

# Prepare columns for display (only those that were originally present and now encoded)
display_columns_list = []
for original_col in categorical_cols_to_process:
    if original_col in df.columns:
        display_columns_list.append(original_col)
    if original_col + '_encoded' in df.columns: # Check for the newly created encoded column
        display_columns_list.append(original_col + '_encoded')

if display_columns_list:
    print("DataFrame head with original and encoded categorical columns:")
    display(df[display_columns_list].head())
else:
    print("No original categorical columns found or encoded columns created.")

# Drop original categorical columns, only if they still exist
# Use errors='ignore' to prevent KeyError if a column is already dropped
df = df.drop(columns=[col for col in categorical_cols_to_process if col in df.columns], errors='ignore')

print("\nOriginal categorical columns dropped (if they existed).")

DataFrame head with original and encoded categorical columns:


,airline_encoded,origin_encoded,destination_encoded,timezone_encoded
0,5,30,37,160
1,5,30,27,330
2,5,30,72,142
3,5,30,65,70
4,5,30,70,307



Original categorical columns dropped (if they existed).
